# 02 — Hillstrom: response baseline and first uplift models (Day 2)

Notebook 01 established that the e-mail was randomly assigned and measured the
**average** effect. This notebook asks the harder question: **can we tell, per
customer, who the e-mail actually persuades?** That is the difference between
targeting everyone (the ATE says it is worth ~+7.7 pp on `visit`) and targeting
*well*.

Vocabulary lives in [`docs/concepts.md`](../../docs/concepts.md). New here:
**response model**, **meta-learners (S/T/X)**, **Qini curve**, **AUUC**,
**uplift@k**.

| Step | Question | Tool |
| --- | --- | --- |
| Response baseline | Does a plain "who will convert" model rank persuadables? | treated-only classifier |
| Meta-learners | Can an uplift model beat that baseline? | S-, T-, X-learners (LightGBM) |
| Comparison | Which ranking captures the most incremental outcome? | Qini curves + ranking table |

**The finding, up front:** on `visit`, the uplift models do **not** cleanly
beat the response baseline, and the S-learner shrinks almost to zero. That is
an honest result, not a bug — §4 and §5 explain why it happens and what it
means for Day 3.

## 0. Setup

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")  # causalml/lightgbm emit benign FutureWarnings

from causalml.inference.meta import BaseSClassifier, BaseTClassifier, BaseXClassifier
from lightgbm import LGBMClassifier, LGBMRegressor

from packages.causal.features import build_features, encode_splits, make_splits
from packages.causal.metrics import random_uplift, ranking_table
from packages.causal.plotting import plot_qini
from packages.datasets.hillstrom import fetch_hillstrom

pd.set_option("display.width", 140)
RANDOM_STATE = 42
print("repo root:", ROOT)

## 1. Load the binary arm and build honest splits

We use the **`mens`** arm chosen on Day 1 (notebook 01 §1) and rank on
**`visit`**, the outcome with enough signal to model (`conversion` has 578
events total — too rare for a first pass).

Two things matter for an honest evaluation:

- **Features never see the outcome or the assignment.** `build_features` takes
  its columns from the loader's `FEATURES`, so `segment`, `treatment` and the
  three outcomes cannot leak in.
- **The split is stratified on `treatment × visit`.** Uplift evaluation
  compares treated against control *within each score bucket*, so a split that
  leaves one arm short of converters makes the Qini curve noisy for reasons
  unrelated to the model.

In [ ]:
df = fetch_hillstrom("mens")
X, y, t = build_features(df, target="visit")

# Encode categoricals to a numeric matrix (fit on train only) because
# causalml coerces X to float internally — LightGBM alone would not need this.
splits, feature_names = encode_splits(make_splits(X, y, t, random_state=RANDOM_STATE))
X_tr, y_tr, t_tr = splits["train"]
X_val, y_val, t_val = splits["val"]
X_te, y_te, t_te = splits["test"]

print(f"rows: train {len(X_tr):,}  val {len(X_val):,}  test {len(X_te):,}")
print(f"visit rate overall: {y.mean():.3%}")
print(f"ATE on test (visit): {y_te[t_te == 1].mean() - y_te[t_te == 0].mean():+.4f}")
print(f"features ({len(feature_names)}): {feature_names}")

## 2. The response baseline — the model that ranks "sure things"

The **response baseline** is a plain classifier trained on **treated customers
only** (the "CTR-style" model). It predicts *who converts after the e-mail*,
not *who converts because of it*. Everyone builds one; it is the bar any
uplift model must clear.

Its blind spot is the project's whole premise. A response model ranks **sure
things** highest — customers with high conversion probability who would have
bought anyway. We can show that directly: check whether the customers the
response model puts at the top also convert at a high rate **in the control
arm**. If they do, their score is tracking *purchasability*, not
*persuadability*.

In [ ]:
# Fit the response model on treated train customers only.
resp_model = LGBMClassifier(
    n_estimators=300, learning_rate=0.03, num_leaves=63,
    min_child_samples=20, random_state=RANDOM_STATE, verbose=-1,
).fit(X_tr[t_tr == 1], y_tr[t_tr == 1])

# Score the *whole* test set (both arms) with it.
resp_score = resp_model.predict_proba(X_te)[:, 1]

# Split the test set into the response model's top and bottom thirds, then
# compare the observed visit rate in TREATED vs CONTROL within each third.
q = pd.qcut(resp_score, 3, labels=["low", "mid", "high"])
probe = pd.DataFrame({"resp_tercile": q, "visit": y_te.values, "treat": t_te.values})
rates = probe.groupby(["resp_tercile", "treat"], observed=True)["visit"].mean().unstack()
rates.columns = ["control", "treated"]
rates["gap (treated - control)"] = rates["treated"] - rates["control"]
print("Visit rate by response-score tercile, split by arm:")
print(rates.round(4))

**Reading §2.** If the response model were ranking persuadables, the *gap*
between treated and control would grow as the score rises. Instead we expect
the **level** to rise in *both* arms together — the top tercile visits often
whether or not it got the e-mail. That is the signature of **sure things**:
high score, low incremental effect. This is why targeting by response score
spends the budget on people who were already going to convert.

(The response model is still a *useful* baseline precisely because it is good
at finding converters — just not at finding *persuadable* ones.)

## 3. The meta-learners — S, T and X

Meta-learners reuse any supervised model to estimate uplift. With LightGBM as
the base learner:

| Learner | How it estimates uplift | Tendency |
| --- | --- | --- |
| **S** (single) | One model on `[X, treatment]`, output the treatment interaction | Pools all data → **shrinks uplift toward zero** |
| **T** (two) | Separate models for treated and control; subtract the predictions | Flexible, but each model sees only half the data |
| **X** (cross) | Imputes per-customer effects from T, then learns a model on them | Often strongest with imbalance; needs a *regressor* for the effect stage |

All three predict a **relative** uplift score (not a probability); only its
ranking matters. We fit on the encoded train split and score the test split.

In [ ]:
def make_clf():
    return LGBMClassifier(
        n_estimators=300, learning_rate=0.03, num_leaves=63,
        min_child_samples=20, random_state=RANDOM_STATE, verbose=-1,
    )

def make_reg():
    return LGBMRegressor(
        n_estimators=300, learning_rate=0.03, num_leaves=63,
        min_child_samples=20, random_state=RANDOM_STATE, verbose=-1,
    )

Xv, tv, yv = X_tr.values, t_tr.values, y_tr.values
Xte = X_te.values

# S-learner: one model, treatment as a feature.
s_learner = BaseSClassifier(make_clf())
s_learner.fit(Xv, tv, yv)
uplift_s = s_learner.predict(Xte).ravel()

# T-learner: one model per arm, subtract.
t_learner = BaseTClassifier(make_clf(), make_clf())
t_learner.fit(Xv, tv, yv)
uplift_t = t_learner.predict(Xte).ravel()

# X-learner: outcome models are classifiers, the effect stage is a regressor.
x_learner = BaseXClassifier(outcome_learner=make_clf(), effect_learner=make_reg())
x_learner.fit(Xv, tv, yv)
uplift_x = x_learner.predict(Xte).ravel()

print(f"S range: {uplift_s.min():.5f} .. {uplift_s.max():.5f}")
print(f"T range: {uplift_t.min():.5f} .. {uplift_t.max():.5f}")
print(f"X range: {uplift_x.min():.5f} .. {uplift_x.max():.5f}")

## 4. Compare: Qini curves and the ranking table

The **Qini curve** plots cumulative incremental outcome against targeting
depth. Sort customers by score, target the top *k*, and accumulate the
treated-minus-control outcome difference among those targeted. A model worth
deploying pulls above the **random** line early; the **perfect** line shows
the ceiling.

**One shape note:** every curve *must* converge at depth 1.0. Once you target
100% of customers, the total treated-minus-control outcome is fixed no matter
the ranking order, so all lines meet at the same endpoint. That is why the
perfect curve drops back down at the very end — it stops ranking
persuadables first and is finally forced to add control converters. The gap
between the perfect peak and the endpoint is the headroom a real model is
trying to capture.

Three numbers summarise it:

- **Qini AUC** — area under that curve, normalised by the perfect line. 0 =
  random, negative = worse than random.
- **AUUC** — area under the raw uplift curve.
- **uplift@k** — incremental outcome per customer, targeting only the top
  `k`. This is the number that multiplies up to a campaign total.

In [ ]:
scores = {
    "s-learner": uplift_s,
    "t-learner": uplift_t,
    "x-learner": uplift_x,
    "response": resp_score,
    "random": random_uplift(y_te.values, t_te.values, random_state=RANDOM_STATE),
}

table = ranking_table(y_te.values, t_te.values, scores)
table

In [ ]:
fig = plot_qini(
    y_te.values, t_te.values, scores,
    title="Qini curves — Day 2 uplift models vs response baseline (visit)",
    name="day2_qini_models_vs_baseline",
)
plt.show()

![Qini curves — Day 2 models vs baseline](../../docs/figures/day2_qini_models_vs_baseline.png)

## 5. Interpretation — what the results actually say

Three observations, in the order they matter:

**1. The uplift signal on `visit` is weak.** Every model sits close to the
random line, and the Qini AUCs are small. That is a property of the data, not
a modelling failure: the Day-1 ATE on `visit` was **+7.76 pp**, but the
heterogeneity around it is small relative to the noise, so there is little
per-customer signal to learn. When we raised model capacity (more trees,
more leaves, fewer samples per leaf), the T-learner moved from *below* the
response baseline to roughly *even* with it — confirming the ceiling is set
by the data, not the learner.

**2. The S-learner shrinks uplift toward zero.** It pools all customers into
one model with `treatment` as just another feature. The easiest way for a
regularised learner to fit that is to learn the *main effect* (visit is
common) and make the treatment interaction tiny. That is why its Qini AUC
lands near zero: it barely ranks anyone by incremental effect. This is the
textbook S-learner failure mode, and it is why the T- and X-learners exist.

**3. The response baseline is embarrassingly competitive.** On `visit` it
matches or beats several uplift models. §2 explains why: its score correlates
with *purchasability*, and because the e-mail lifts everyone somewhat, the
highest-purchasability customers also carry a lot of the total incremental
outcome. That advantage disappears the moment we optimise **profit** rather
than outcome — sure things cost full contact price for little margin
recovery. Day 4 makes that trade-off explicit.

## 6. What this sets up for Day 3

- **Try a regression target.** `visit` may be too coarse. The DR-learner on
  `spend` (real dollars) could separate persuadables better — a customer
  convinced to spend $300 is worth detecting even if they would have visited
  anyway.
- **DR-learner and causal forest.** Both use more of the data than the T/X
  imputation step and report effect *intervals*, which the optimizer needs.
- **Calibration.** These scores are rankings, not effect sizes. Before any
  profit figure is trusted, we check predicted-vs-observed uplift by decile.
- **Stability.** One split can flatter a model. Day 3 repeats over 10 seeds
  and reports mean ± spread.

**Day-2 takeaway to carry forward:** an uplift model does not automatically
beat a response model on *outcome* metrics. The win has to come from the
**decision layer** — profit under a budget — which is exactly what Day 4 is
for. Concepts and these numbers are mirrored in
[`docs/concepts.md`](../../docs/concepts.md).